In [1]:
from typing import List
import numpy as np
import pandas as pd
import warnings
import logging
import os
import shutil
import json
from sklearn.metrics import mean_squared_error
import torch
from tqdm import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from collections import Counter
import spacy
import re
from autocorrect import Speller
from spellchecker import SpellChecker
import lightgbm as lgb

warnings.simplefilter("ignore")
logging.disable(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
tqdm.pandas()

In [2]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    
seed_everything(seed=42)

In [3]:
class CFG:
    model_name="microsoft/deberta-v3-large"
    learning_rate=0.000016   #0.000015
    weight_decay=0.05        #0.02
    hidden_dropout_prob=0
    attention_probs_dropout_prob=0
    num_train_epochs=3
    n_splits=4
    batch_size=2
    random_seed=42
    save_steps=100
    max_length=512

## Preprocess

[Using features]

- Text Length
- Length Ratio
- Word Overlap
- N-grams Co-occurrence
  - count
  - ratio
- Quotes Overlap
- Grammar Check
  - spelling: pyspellchecker


In [4]:
class Preprocessor:
    def __init__(self, 
                model_name: str,
                ) -> None:
#         self.tokenizer = AutoTokenizer.from_pretrained(model_name)
#         self.twd = TreebankWordDetokenizer()
        self.STOP_WORDS = set(stopwords.words('english'))
        
        self.spacy_ner_model = spacy.load('en_core_web_sm',)
        self.speller = Speller(lang='en')
        self.spellchecker = SpellChecker() 
        
    def word_overlap_count(self, row):
        """ intersection(prompt_text, text) """        
        def check_is_stop_word(word):
            return word in self.STOP_WORDS
        
        prompt_words = row['prompt_tokens']
        summary_words = row['summary_tokens']
        if self.STOP_WORDS:
            prompt_words = list(filter(check_is_stop_word, prompt_words))
            summary_words = list(filter(check_is_stop_word, summary_words))
        return len(set(prompt_words).intersection(set(summary_words)))
            
    def ngrams(self, token, n):
        # Use the zip function to help us generate n-grams
        # Concatentate the tokens into ngrams and return
        ngrams = zip(*[token[i:] for i in range(n)])
        return [" ".join(ngram) for ngram in ngrams]

    def ngram_co_occurrence(self, row, n: int) -> int:
        # Tokenize the original text and summary into words
        original_tokens = row['prompt_tokens']
        summary_tokens = row['summary_tokens']

        # Generate n-grams for the original text and summary
        original_ngrams = set(self.ngrams(original_tokens, n))
        summary_ngrams = set(self.ngrams(summary_tokens, n))

        # Calculate the number of common n-grams
        common_ngrams = original_ngrams.intersection(summary_ngrams)
        return len(common_ngrams)
    
    def ner_overlap_count(self, row, mode:str):
        model = self.spacy_ner_model
        def clean_ners(ner_list):
            return set([(ner[0].lower(), ner[1]) for ner in ner_list])
        prompt = model(row['prompt_text'])
        summary = model(row['text'])

        if "spacy" in str(model):
            prompt_ner = set([(token.text, token.label_) for token in prompt.ents])
            summary_ner = set([(token.text, token.label_) for token in summary.ents])
        elif "stanza" in str(model):
            prompt_ner = set([(token.text, token.type) for token in prompt.ents])
            summary_ner = set([(token.text, token.type) for token in summary.ents])
        else:
            raise Exception("Model not supported")

        prompt_ner = clean_ners(prompt_ner)
        summary_ner = clean_ners(summary_ner)

        intersecting_ners = prompt_ner.intersection(summary_ner)
        
        ner_dict = dict(Counter([ner[1] for ner in intersecting_ners]))
        
        if mode == "train":
            return ner_dict
        elif mode == "test":
            return {key: ner_dict.get(key) for key in self.ner_keys}

    
    def quotes_count(self, row):
        summary = row['text']
        text = row['prompt_text']
        quotes_from_summary = re.findall(r'"([^"]*)"', summary)
        if len(quotes_from_summary)>0:
            return [quote in text for quote in quotes_from_summary].count(True)
        else:
            return 0

    def spelling(self, text):
        
        wordlist=text.split()
        amount_miss = len(list(self.spellchecker.unknown(wordlist)))

        return amount_miss
    
    def add_spelling_dictionary(self, tokens: List[str]) -> List[str]:
        """dictionary update for pyspell checker and autocorrect"""
        self.spellchecker.word_frequency.load_words(tokens)
        self.speller.nlp_data.update({token:1000 for token in tokens})
    
    def run(self, 
            dataframe: pd.DataFrame,
            mode:str
        ) -> pd.DataFrame:
        
        # before merge preprocess
        dataframe["prompt_tokens"] = dataframe["prompt_text"].progress_apply(lambda x: word_tokenize(x))
        dataframe["summary_tokens"] = dataframe["text"].progress_apply(lambda x: word_tokenize(x))
        
        dataframe["prompt_length"] = dataframe["prompt_tokens"].progress_apply(lambda x: len(x))
        dataframe["summary_length"] = dataframe["summary_tokens"].progress_apply(lambda x: len(x))
        
        dataframe["prompt_tokens"].progress_apply(lambda x: self.add_spelling_dictionary(x)) # Add prompt tokens into spelling checker dictionary
#         dataframe["fixed_summary_text"] = dataframe["text"].progress_apply(lambda x: self.speller(x)) # fix misspelling
        dataframe["spelling_err_num"] = dataframe["text"].progress_apply(self.spelling) # count misspelling
        
        dataframe['length_ratio'] = dataframe['summary_length'] / dataframe['prompt_length']
        
        dataframe['word_overlap_count'] = dataframe.progress_apply(self.word_overlap_count, axis=1)
        dataframe['bigram_overlap_count'] = dataframe.progress_apply(self.ngram_co_occurrence,args=(2,), axis=1 )
        dataframe['bigram_overlap_ratio'] = dataframe['bigram_overlap_count'] / (dataframe['summary_length'] - 1)
        dataframe['trigram_overlap_count'] = dataframe.progress_apply(self.ngram_co_occurrence, args=(3,), axis=1)
        dataframe['trigram_overlap_ratio'] = dataframe['trigram_overlap_count'] / (dataframe['summary_length'] - 2)
        
        dataframe['quotes_count'] = dataframe.progress_apply(self.quotes_count, axis=1)
        
        
        return dataframe.drop(columns=["summary_tokens", "prompt_tokens"])
    
preprocessor = Preprocessor(model_name=CFG.model_name)

## Model Function Definition

In [5]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    rmse = mean_squared_error(labels, predictions, squared=False)
    return {"rmse": rmse}

def compute_mcrmse(eval_pred):
    """
    Calculates mean columnwise root mean squared error
    https://www.kaggle.com/competitions/commonlit-evaluate-student-summaries/overview/evaluation
    """
    preds, labels = eval_pred

    col_rmse = np.sqrt(np.mean((preds - labels) ** 2, axis=0))
    mcrmse = np.mean(col_rmse)

    return {
        "content_rmse": col_rmse[0],
        "wording_rmse": col_rmse[1],
        "mcrmse": mcrmse,
    }

def compt_score(content_true, content_pred, wording_true, wording_pred):
    content_score = mean_squared_error(content_true, content_pred)**(1/2)
    wording_score = mean_squared_error(wording_true, wording_pred)**(1/2)
    
    return (content_score + wording_score)/2

In [6]:
# # exp_name = "c1-442-d3lm21p4-512-8-22-5e51e3-C099050d0-rmse"
# exp_name = "c1-442-d3bm21p4-512-12-22-6e51e3-C099020d0-rmse"
# folds = 4

# df=[]
# for idx, fold in enumerate(range(folds)):
#     print("Fold : ", fold)
#     fold_df = pd.read_csv(f"../../output/commonlit-evaluate-student-summaries//{exp_name}/predictions_fold{fold}.csv")
# #     fold_df["fold"] = fold
#     df.append(fold_df)
# df = pd.concat(df, axis=0)
# df.rename(columns = {'content_predictions':'content_pred', 'wording_predictions':'wording_pred'}, inplace = True)
# df.head(2)

In [7]:
models = [
        'c1-442-d3bm21p4-512-12-22-6e51e3-C099020d0-rmse',
        'c1-442-d3lm21p4-512-8-22-5e51e3-C099050d0-rmse',
        'c1-442-dbm21p4-512-12-22-6e51e3-C099020d0-rmse',
        'c1-442-dlm21p4-512-6-22-5e51e3-C099050d0-rmse',
        'c1-442-rbm21p4-512-12-22-6e51e3-C099020d0-rmse',
        'c1-442-rlm21p4-512-12-22-5e51e3-C099050d0-rmse',
        ]
label_cols = ["content", "wording"]
n_folds=4

data_set_df_list=[]
for i in models:
    folds_list=[]
    for fold in range(n_folds):
        folds_list.append(pd.read_csv(f"../../output/commonlit-evaluate-student-summaries/{i}/predictions_fold{fold}.csv"))
    fold_df = pd.concat(folds_list, axis=0)
    data_set_df_list.append(fold_df)
    

final_df = data_set_df_list[0]
for idx, df in enumerate(data_set_df_list):
    for label in label_cols:
        final_df[f'{label}_predictions_{idx}'] = final_df['student_id'].map(df.set_index('student_id')[f'{label}_predictions'])
final_df.head(2)

,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold,prompt_title_processed,...,content_predictions_1,wording_predictions_1,content_predictions_2,wording_predictions_2,content_predictions_3,wording_predictions_3,content_predictions_4,wording_predictions_4,content_predictions_5,wording_predictions_5
0,ad7245db300c,39c16e,The main person is a likable person that is ne...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,...,-1.481757,-1.436132,-1.365821,-1.370134,-1.514836,-1.394568,-1.621638,-1.505012,-1.404012,-1.308822
1,ac8891e90289,39c16e,a complex twisting plot that makes you feel pi...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,...,-1.517863,-1.606803,-1.619965,-1.567841,-1.470178,-1.331284,-1.605907,-1.512542,-1.325764,-1.233075


In [8]:
final_df = preprocessor.run(final_df, mode="train")
final_df.head(2)

100%|██████████| 7165/7165 [00:00<00:00, 101479.67it/s]


,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold,prompt_title_processed,...,prompt_length,summary_length,spelling_err_num,length_ratio,word_overlap_count,bigram_overlap_count,bigram_overlap_ratio,trigram_overlap_count,trigram_overlap_ratio,quotes_count
0,ad7245db300c,39c16e,The main person is a likable person that is ne...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,...,699,26,0,0.037196,10,5,0.200000,3,0.125,0
1,ac8891e90289,39c16e,a complex twisting plot that makes you feel pi...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,...,699,27,1,0.038627,7,5,0.192308,0,0.000,0


In [9]:
weights_0 = [0.4156817045344159, 0.6319900305212296]
weights_1 = [0.48448299301024383, 0.7637415141040962, -0.2010484484370789]
weights_2 = [0.4736402095435008, 0.6565757058607872, -0.2415362737222119, 0.16108816116309504]
weights_3 = [0.4651089427984977, 0.7008473171929651, -0.21157183017002118, 0.2520641498519185, -0.14657945056310245]
weights_4 = [0.44156833981958465, 0.7365164665441468, -0.15709296775105847, 0.2965906573183049, -0.13191138230524024, -0.12457739566343137]

for label in label_cols:
    final_df[f"{label}_pred_0"] = weights_0[0] * final_df[f"{label}_predictions_0"] + \
                                  weights_0[1] * final_df[f"{label}_predictions_1"] 
    
    final_df[f"{label}_pred_1"] = weights_1[0] * final_df[f"{label}_predictions_0"] + \
                                  weights_1[1] * final_df[f"{label}_predictions_1"] + \
                                  weights_1[2] * final_df[f"{label}_predictions_2"]
    
    final_df[f"{label}_pred_2"] = weights_2[0] * final_df[f"{label}_predictions_0"] + \
                                  weights_2[1] * final_df[f"{label}_predictions_1"] + \
                                  weights_2[2] * final_df[f"{label}_predictions_2"] + \
                                  weights_2[3] * final_df[f"{label}_predictions_3"] 
    
    final_df[f"{label}_pred_3"] = weights_3[0] * final_df[f"{label}_predictions_0"] + \
                                  weights_3[1] * final_df[f"{label}_predictions_1"] + \
                                  weights_3[2] * final_df[f"{label}_predictions_2"] + \
                                  weights_3[3] * final_df[f"{label}_predictions_3"] + \
                                  weights_3[4] * final_df[f"{label}_predictions_4"] 
    
    final_df[f"{label}_pred_4"] = weights_4[0] * final_df[f"{label}_predictions_0"] + \
                                  weights_4[1] * final_df[f"{label}_predictions_1"] + \
                                  weights_4[2] * final_df[f"{label}_predictions_2"] + \
                                  weights_4[3] * final_df[f"{label}_predictions_3"] + \
                                  weights_4[4] * final_df[f"{label}_predictions_4"] + \
                                  weights_4[5] * final_df[f"{label}_predictions_5"]
    

In [10]:
final_df.columns

Index(['student_id', 'prompt_id', 'text', 'content', 'wording',
       'prompt_question', 'prompt_title', 'prompt_text', 'fold',
       'prompt_title_processed', 'prompt_question_processed',
       'prompt_text_processed', 'text_processed', 'full_text_processed',
       'full_text_len', 'content_predictions', 'wording_predictions',
       'content_predictions_0', 'wording_predictions_0',
       'content_predictions_1', 'wording_predictions_1',
       'content_predictions_2', 'wording_predictions_2',
       'content_predictions_3', 'wording_predictions_3',
       'content_predictions_4', 'wording_predictions_4',
       'content_predictions_5', 'wording_predictions_5', 'prompt_length',
       'summary_length', 'spelling_err_num', 'length_ratio',
       'word_overlap_count', 'bigram_overlap_count', 'bigram_overlap_ratio',
       'trigram_overlap_count', 'trigram_overlap_ratio', 'quotes_count',
       'content_pred_0', 'content_pred_1', 'content_pred_2', 'content_pred_3',
       'content_p

In [11]:
def count_capital_words(text):
    return sum(map(str.isupper,text.split()))

def count_punctuations(text):
    punctuations='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~'
    d=dict()
    for i in punctuations:
        d[str(i)+' count']=text.count(i)
    return d 


def count_sent(text):
    return len(nltk.sent_tokenize(text))


def count_unique_words(text):
    return len(set(text.split()))

def count_htags(text):
    x = re.findall(r'(#w[A-Za-z0-9]*)', text)
    return len(x) 


# def count_mentions(text):
#     x = re.findall(r'(@w[A-Za-z0-9]*)', text)
#     return len(x)


def count_stopwords(text):
    stop_words = set(stopwords.words('english'))  
    word_tokens = word_tokenize(text)
    stopwords_x = [w for w in word_tokens if w in stop_words]
    return len(stopwords_x)

def count_chars(text):
    return len(text)

def count_words(text):
    return len(text.split())

In [12]:
final_df["count_chars"] = final_df["text"].progress_apply(lambda x: count_chars(x))
final_df["count_words"] = final_df["text"].progress_apply(lambda x: count_words(x))
final_df["count_capital_words"] = final_df["text"].progress_apply(lambda x: count_capital_words(x))
# final_df["count_punctuations"] = final_df["text"].progress_apply(lambda x: count_punctuations(x))
final_df["count_sent"] = final_df["text"].progress_apply(lambda x: count_sent(x))
final_df["count_unique_words"] = final_df["text"].progress_apply(lambda x: count_unique_words(x))
final_df["count_stopwords"] = final_df["text"].progress_apply(lambda x: count_stopwords(x))

final_df['avg_wordlength'] = final_df['count_chars']/final_df['count_words']
final_df['avg_sentlength'] = final_df['count_words']/final_df['count_sent']
final_df['unique_vs_words'] = final_df['count_unique_words']/final_df['count_words']
final_df['stopwords_vs_words'] = final_df['count_stopwords']/final_df['count_words']

100%|██████████| 7165/7165 [00:03<00:00, 2339.13it/s]


## LGBM model

In [20]:
targets = ["content", "wording"]

want_content_columns = ['full_text_len', 
                        'prompt_length',
                        'summary_length', 
                        'spelling_err_num', 
        #                 'length_ratio',
                        'word_overlap_count', 
                        'bigram_overlap_count', 
                        'bigram_overlap_ratio',
                        'trigram_overlap_count', 
                        'trigram_overlap_ratio', 
                        'quotes_count',

                        "count_chars",
                        "count_words",
                        "count_capital_words",
        #                 "count_punctuations",
                        "count_sent",
                        "count_unique_words",
                        "count_stopwords",

                        "avg_wordlength",
                        "avg_sentlength",
                        "unique_vs_words",
                        "stopwords_vs_words",
                
                        'content_predictions_0', 
                        'content_predictions_1', 
                        'content_predictions_2', 
                        'content_predictions_3', 
                        'content_predictions_4', 
#                         'content_predictions_5', 

                
                        'content_pred_0', 
                        'content_pred_1', 
                        'content_pred_2', 
                        'content_pred_3',
#                         'content_pred_4',
                       ]


want_wording_columns = ['full_text_len', 
                        'prompt_length',
                        'summary_length', 
                        'spelling_err_num', 
        #                 'length_ratio',
                        'word_overlap_count', 
                        'bigram_overlap_count', 
                        'bigram_overlap_ratio',
                        'trigram_overlap_count', 
                        'trigram_overlap_ratio', 
                        'quotes_count',

                        "count_chars",
                        "count_words",
                        "count_capital_words",
        #                 "count_punctuations",
                        "count_sent",
                        "count_unique_words",
                        "count_stopwords",

                        "avg_wordlength",
                        "avg_sentlength",
                        "unique_vs_words",
                        "stopwords_vs_words",

                        'wording_predictions_0',
                        'wording_predictions_1',
                        'wording_predictions_2',
                        'wording_predictions_3',
                        'wording_predictions_4',
#                         'wording_predictions_5',

                        'wording_pred_0',
                        'wording_pred_1',
                        'wording_pred_2',
                        'wording_pred_3',
#                         'wording_pred_4',
               ]
drop_columns = ["fold"] + targets

In [21]:
train = final_df

In [22]:
# train = final_df[want_columns]
# train.head(2)

In [23]:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# tf_dict = {}
model_dict = {}

for target in targets:
    models = []
#     scalers = []
    for fold in range(4):
        if target == "content":
            want_columns = want_content_columns  
        elif target == "wording":            
            want_columns = want_wording_columns  

        X_train_cv = train[train["fold"] != fold].drop(columns=drop_columns)
        y_train_cv = train[train["fold"] != fold][target]
        print(X_train_cv.columns)

        X_eval_cv = train[train["fold"] == fold].drop(columns=drop_columns)
        y_eval_cv = train[train["fold"] == fold][target]
        
        X_train_cv = X_train_cv[want_columns]
        X_eval_cv = X_eval_cv[want_columns]

        dtrain = lgb.Dataset(X_train_cv, label=y_train_cv)
        dval = lgb.Dataset(X_eval_cv, label=y_eval_cv)

        params = {
            'boosting_type': 'gbdt',
            'random_state': 42,
            'objective': 'regression',
            'metric': 'rmse',
#             'learning_rate': 0.045,
            'max_depth': 3,  #3
            'lambda_l1': 0.0,
            'lambda_l2': 0.015
        }

        evaluation_results = {}
        model = lgb.train(params,
                          num_boost_round=10000,
                            #categorical_feature = categorical_features,
                          valid_names=['train', 'valid'],
                          train_set=dtrain,
                          valid_sets=dval,
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=30, verbose=False),
                               lgb.log_evaluation(100),
                              lgb.callback.record_evaluation(evaluation_results)
                            ],
                          )
        models.append(model)
#         scalers.append(scaler)
    
    model_dict[target] = models
#     tf_dict[target] = scalers

Index(['student_id', 'prompt_id', 'text', 'prompt_question', 'prompt_title',
       'prompt_text', 'prompt_title_processed', 'prompt_question_processed',
       'prompt_text_processed', 'text_processed', 'full_text_processed',
       'full_text_len', 'content_predictions', 'wording_predictions',
       'content_predictions_0', 'wording_predictions_0',
       'content_predictions_1', 'wording_predictions_1',
       'content_predictions_2', 'wording_predictions_2',
       'content_predictions_3', 'wording_predictions_3',
       'content_predictions_4', 'wording_predictions_4',
       'content_predictions_5', 'wording_predictions_5', 'prompt_length',
       'summary_length', 'spelling_err_num', 'length_ratio',
       'word_overlap_count', 'bigram_overlap_count', 'bigram_overlap_ratio',
       'trigram_overlap_count', 'trigram_overlap_ratio', 'quotes_count',
       'content_pred_0', 'content_pred_1', 'content_pred_2', 'content_pred_3',
       'content_pred_4', 'wording_pred_0', 'wording_pr

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000373 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5501
[LightGBM] [Info] Number of data points in the train set: 5169, number of used features: 29
[LightGBM] [Info] Start training from score 0.028040
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

## CV Score

In [24]:
# cv
rmses = []

for target in targets:
    models = model_dict[target]

    if target == "content":
        want_columns = want_content_columns  
    elif target == "wording":            
        want_columns = want_wording_columns 
        
    preds = []
    trues = []
    
    for fold, model in enumerate(models):
        X_eval_cv = train[train["fold"] == fold].drop(columns=drop_columns)
        y_eval_cv = train[train["fold"] == fold][target]

        X_eval_cv = X_eval_cv[want_columns]
        
        pred = model.predict(X_eval_cv)

        trues.extend(y_eval_cv)
        preds.extend(pred)
        
        model.save_model(f'../../output/commonlit-evaluate-student-summaries/lgb/ensemble/fold_{target}_{fold}.txt')
        
    rmse = np.sqrt(mean_squared_error(trues, preds))
    print(f"{target}_rmse : {rmse}")
    rmses = rmses + [rmse]

print(f"mcrmse : {sum(rmses) / len(rmses)}")

content_rmse : 0.42837551126437384
wording_rmse : 0.5477104762521192
mcrmse : 0.48804299375824656


In [25]:
0.48922931325831115

0.48922931325831115

In [26]:
0.4883172126017351
0.48832903366482316

0.48804299375824656

0.48804299375824656